# Notebook de obtención, limpieza y transformación de datos — Fase 2

**Proyecto:** Mantenimiento predictivo de maquinaria industrial mediante AI4I 2020  
**Asignatura:** MCDI500 — Programación para la Ciencia de Datos  
**Grupo:** 9  
**Estudiante:** Alexander Sepulveda Ormazabal  

## ¿Qué es un pipeline de preprocesamiento?

Un pipeline de preprocesamiento es una secuencia ordenada de operaciones que
permite transformar los datos originales en un conjunto consistente y adecuado
para el análisis posterior.

## Flujo de trabajo de la Fase 2

El procesamiento seguirá una secuencia definida:

**Obtener → Explorar → Limpiar → Transformar → Escalar → Validar → Guardar**

Se utilizará como entrada el resultado de la etapa anterior. Si una
validación detecta un problema, el notebook se detendrá para evitar que un error
se propague al resto del proceso.

El archivo original se leerá desde:

`F1/data/raw/ai4i2020.csv`

El resultado procesado se guardará en:

`F2/data/processed/ai4i2020_procesado.csv`

Cada etapa será explicada, ejecutada y verificada. El archivo original se
conservará sin modificaciones y los resultados procesados se guardarán en una
ubicación diferente, permitiendo mantener la trazabilidad del proyecto.

## Introducción

Las fallas inesperadas en maquinaria industrial pueden provocar interrupciones
operacionales, pérdidas de productividad y mayores necesidades de mantenimiento.
El conjunto de datos AI4I 2020 contiene 10.000 registros sobre condiciones de
operación de maquinaria, incluyendo temperatura del aire, temperatura del
proceso, velocidad de rotación, torque y desgaste de herramienta. También
registra la ocurrencia de fallas y sus posibles modos.

Durante la Fase 1 se definió la problemática, se formularon los objetivos del
proyecto, se configuró el entorno reproducible y se documentó la estructura del
conjunto de datos. La Fase 2 desarrolla la implementación técnica necesaria para
obtener, explorar, limpiar, transformar y validar estos datos.

El procesamiento se realizará sobre una copia del conjunto original. Las
decisiones de limpieza y transformación se justificarán mediante los resultados
observados, evitando eliminar o modificar registros sin evidencia técnica.

## Objetivo general de la Fase 2

Construir y validar un pipeline reproducible para obtener, explorar, limpiar y
transformar el conjunto de datos AI4I 2020, generando datos procesados,
trazables y adecuados para las etapas posteriores de análisis y modelamiento.

## Objetivos específicos

1. Cargar el conjunto de datos original y verificar su procedencia, estructura
   e integridad.
2. Examinar los tipos de variables, valores faltantes, duplicados, categorías,
   rangos y distribución de la variable objetivo.
3. Aplicar decisiones justificadas de limpieza y selección de variables,
   conservando intacto el archivo original.
4. Codificar la variable categórica `Type` y comparar alternativas de
   escalamiento para las variables operacionales.
5. Excluir los identificadores y los indicadores de modos de falla del conjunto
   de predictores para evitar información irrelevante o fuga de información.
6. Validar el dataset resultante mediante casos normales, casos límite y
   excepciones.
7. Guardar los resultados procesados y documentar las transformaciones
   realizadas.

## Variables principales

El conjunto original contiene:

- Identificadores: `UDI` y `Product ID`.
- Tipo de producto: `Type`.
- Condiciones operacionales: temperaturas, velocidad de rotación, torque y
  desgaste de herramienta.
- Variable objetivo: `Machine failure`.
- Modos de falla: `TWF`, `HDF`, `PWF`, `OSF` y `RNF`.

Los identificadores no se utilizarán como predictores. Los modos de falla
tampoco se utilizarán para predecir `Machine failure`, porque contienen
información directamente relacionada con la ocurrencia de la falla y podrían
generar fuga de información.


## Librerías utilizadas y reproducibilidad

Para desarrollar el pipeline se utilizarán las siguientes herramientas:

| Herramienta | Uso en el proyecto |
|---|---|
| NumPy | Operaciones numéricas y comprobaciones estadísticas. |
| Pandas | Carga, exploración y transformación de datos. |
| Matplotlib | Gráficos de control y comparación. |
| scikit-learn | Codificación y escalamiento de variables. |
| pathlib | Construcción de rutas reproducibles. |
| hashlib | Verificación de la integridad del archivo original. |
| json | Exportación de metadatos y decisiones técnicas. |

Se utilizará la semilla `42` para reproducir los procedimientos que incorporen
aleatoriedad. También se registrarán el intérprete y las versiones de las
principales librerías.

In [ ]:
import sys
import json
import hashlib
import platform
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    MinMaxScaler,
    RobustScaler,
)

# Semilla común para todo el proyecto.
SEMILLA = 42
np.random.seed(SEMILLA)

# Mostrar los gráficos dentro del notebook.
%matplotlib inline

print("Cuaderno de la Fase 2 · Grupo 9 ·", date.today().isoformat())
print("Semilla declarada:", SEMILLA)
print("Intérprete:", sys.executable)
print("Sistema:", platform.platform())
print("Carpeta de trabajo:", Path.cwd())
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

## Configuración de las rutas del proyecto

Las rutas no deben depender de una carpeta personal como `C:/Users/...`, porque
esa dirección solo funcionaría en un computador.

El notebook buscará automáticamente la raíz del repositorio mediante dos
elementos conocidos: `README.md` y el archivo original `ai4i2020.csv`. Después
construirá las rutas de entrada y salida a partir de esa raíz.

El archivo original se conservará en F1. Las carpetas de salida de F2 se crearán
automáticamente si todavía no existen.

In [ ]:
# Carpeta desde la cual está ejecutándose actualmente el notebook.
ACTUAL = Path.cwd().resolve()

# Buscar hacia arriba hasta encontrar la raíz real del repositorio.
RAIZ = next(
    (
        ruta
        for ruta in [ACTUAL, *ACTUAL.parents]
        if (ruta / "README.md").is_file()
        and (ruta / "F1" / "data" / "raw" / "ai4i2020.csv").is_file()
    ),
    None,
)

if RAIZ is None:
    raise FileNotFoundError(
        "No se encontró la raíz del proyecto ni el archivo "
        "'F1/data/raw/ai4i2020.csv'."
    )

# Entrada: archivo original de la Fase 1.
DIR_CRUDO = RAIZ / "F1" / "data" / "raw"
ARCHIVO_CRUDO = DIR_CRUDO / "ai4i2020.csv"

# Salidas: carpetas pertenecientes a la Fase 2.
DIR_PROCESADO = RAIZ / "F2" / "data" / "processed"
DIR_DOCS = RAIZ / "F2" / "docs"
DIR_SRC = RAIZ / "F2" / "src"
DIR_NOTEBOOKS = RAIZ / "F2" / "notebooks"

# Crear las carpetas de salida si todavía no existen.
for carpeta in (DIR_PROCESADO, DIR_DOCS, DIR_SRC, DIR_NOTEBOOKS):
    carpeta.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", RAIZ)
print("Archivo original:", ARCHIVO_CRUDO)
print("¿Existe el archivo?:", ARCHIVO_CRUDO.is_file())
print("Datos procesados:", DIR_PROCESADO)
print("Documentación:", DIR_DOCS)
print("Código reutilizable:", DIR_SRC)

## 1. Obtención de los datos

### 1.1 Procedencia e integridad del archivo original

| Campo | Información |
|---|---|
| Nombre | AI4I 2020 Predictive Maintenance Dataset |
| Autor | Stephan Matzka |
| Repositorio | UCI Machine Learning Repository |
| Enlace | https://archive.ics.uci.edu/dataset/601/ai4i+2020+predictive+maintenance+dataset |
| DOI | https://doi.org/10.24432/C5HS5C |
| Licencia | Creative Commons Attribution 4.0 International |
| Estructura esperada | 10.000 filas y 14 columnas |
| Unidad de observación | Registro de condiciones operacionales de una máquina |

**Referencia APA 7:**

Matzka, S. (2020). *AI4I 2020 Predictive Maintenance Dataset*
[Conjunto de datos]. UCI Machine Learning Repository.
https://doi.org/10.24432/C5HS5C

Para mantener la trazabilidad se calculará el hash SHA-256 del archivo original.
Este valor funciona como una huella digital: si el contenido del CSV cambia,
también cambiará su hash.

In [ ]:
def calcular_sha256(ruta, tamano_bloque=65536):
    """
    Calcula la huella SHA-256 de un archivo.

    Parámetros
    ----------
    ruta : str o pathlib.Path
        Archivo cuya integridad se desea comprobar.
    tamano_bloque : int
        Cantidad de bytes leídos en cada iteración.

    Retorna
    -------
    str
        Hash SHA-256 expresado como texto hexadecimal.
    """
    ruta = Path(ruta)

    if not ruta.is_file():
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")

    algoritmo = hashlib.sha256()

    with ruta.open("rb") as archivo:
        for bloque in iter(
            lambda: archivo.read(tamano_bloque),
            b"",
        ):
            algoritmo.update(bloque)

    return algoritmo.hexdigest()


HASH_ARCHIVO_CRUDO = calcular_sha256(ARCHIVO_CRUDO)

print("Archivo:", ARCHIVO_CRUDO.name)
print("Tamaño:", ARCHIVO_CRUDO.stat().st_size, "bytes")
print("SHA-256:", HASH_ARCHIVO_CRUDO)
print("Longitud del hash:", len(HASH_ARCHIVO_CRUDO))

### 1.2 Carga y verificación del archivo CSV

El archivo original se cargará mediante una función que comprobará tres
condiciones antes de continuar:

1. El archivo debe existir.
2. Debe contener registros.
3. Sus columnas deben coincidir con la estructura documentada de AI4I 2020.

La función devolverá un DataFrame, que es la estructura de Pandas utilizada para
trabajar con datos organizados en filas y columnas. Se conservará una copia
denominada `df_crudo` y se creará otra copia llamada `df` para las operaciones
posteriores.

In [ ]:
COLUMNAS_ESPERADAS = [
    "UDI",
    "Product ID",
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Machine failure",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]


def cargar_datos(ruta, columnas_esperadas):
    """
    Carga un CSV y verifica su estructura básica.
    """
    ruta = Path(ruta)

    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo: {ruta}"
        )

    try:
        datos = pd.read_csv(
            ruta,
            encoding="utf-8-sig",
        )
    except pd.errors.EmptyDataError as error:
        raise ValueError(
            "El archivo CSV está vacío."
        ) from error

    if datos.empty:
        raise ValueError(
            "El archivo contiene columnas, pero no tiene registros."
        )

    columnas_observadas = datos.columns.tolist()

    if columnas_observadas != columnas_esperadas:
        faltantes = [
            columna
            for columna in columnas_esperadas
            if columna not in columnas_observadas
        ]
        adicionales = [
            columna
            for columna in columnas_observadas
            if columna not in columnas_esperadas
        ]

        raise ValueError(
            "La estructura no coincide con AI4I 2020. "
            f"Faltantes: {faltantes}. "
            f"Adicionales: {adicionales}."
        )

    print("Datos cargados correctamente")
    print("Filas:", datos.shape[0])
    print("Columnas:", datos.shape[1])

    return datos.copy(deep=True)


df_crudo = cargar_datos(
    ARCHIVO_CRUDO,
    COLUMNAS_ESPERADAS,
)

# Esta es la copia sobre la que trabajaremos.
df = df_crudo.copy(deep=True)

print("Copia de trabajo creada:", df.shape)
df.head()

## 2. Exploración inicial

### 2.1 Estructura y tipos de datos

Antes de transformar el conjunto se construirá una línea base de su estado
original. Se revisarán sus dimensiones, tipos de almacenamiento, cantidad de
valores no nulos, valores faltantes y valores únicos.

Esta revisión permite detectar problemas como columnas vacías, variables
numéricas cargadas como texto o categorías con una cantidad inesperada de
valores.

In [ ]:
def resumir_estructura(datos):
    """
    Construye un resumen de la estructura de un DataFrame.
    """
    resumen = pd.DataFrame({
        "variable": datos.columns,
        "tipo_dato": [
            str(datos[columna].dtype)
            for columna in datos.columns
        ],
        "no_nulos": [
            int(datos[columna].notna().sum())
            for columna in datos.columns
        ],
        "nulos": [
            int(datos[columna].isna().sum())
            for columna in datos.columns
        ],
        "valores_unicos": [
            int(datos[columna].nunique(dropna=True))
            for columna in datos.columns
        ],
    })

    resumen["porcentaje_nulos"] = (
        resumen["nulos"] / len(datos) * 100
    ).round(2)

    return resumen


RESUMEN_ESTRUCTURA = resumir_estructura(df_crudo)

print("Dimensiones:", df_crudo.shape)
print("Memoria utilizada:", round(
    df_crudo.memory_usage(deep=True).sum() / 1024,
    2,
), "KB")

RESUMEN_ESTRUCTURA

### 2.2 Valores faltantes y filas duplicadas

Los valores faltantes y las filas duplicadas pueden afectar los cálculos y
producir resultados sesgados. Antes de aplicar cualquier corrección se
cuantificarán ambos problemas.

Si no se detectan valores faltantes, no se aplicará imputación. Imputar datos
que no faltan sería una transformación innecesaria.

Las filas duplicadas exactas se revisarán utilizando todas las columnas. Los
identificadores también se comprobarán por separado, porque `UDI` y
`Product ID` deberían ser únicos.


In [ ]:
def diagnosticar_calidad_basica(datos):
    """
    Resume valores faltantes, duplicados e identificadores repetidos.
    """
    return pd.DataFrame([{
        "filas": int(datos.shape[0]),
        "columnas": int(datos.shape[1]),
        "nulos_totales": int(datos.isna().sum().sum()),
        "filas_con_nulos": int(datos.isna().any(axis=1).sum()),
        "duplicados_exactos": int(datos.duplicated().sum()),
        "udi_repetidos": int(datos["UDI"].duplicated().sum()),
        "product_id_repetidos": int(
            datos["Product ID"].duplicated().sum()
        ),
    }])


CALIDAD_BASICA = diagnosticar_calidad_basica(df_crudo)
CALIDAD_BASICA

El diagnóstico se interpretará mediante condiciones. Esto permite que el
notebook informe la decisión correspondiente según los resultados observados,
en vez de asumir de antemano que los datos necesitan correcciones.

In [ ]:
nulos_totales = int(df_crudo.isna().sum().sum())
duplicados_exactos = int(df_crudo.duplicated().sum())

if nulos_totales == 0:
    print(
        "[OK] No se encontraron valores faltantes. "
        "No corresponde aplicar imputación."
    )
else:
    print(
        f"[REVISAR] Se encontraron {nulos_totales} valores faltantes."
    )

if duplicados_exactos == 0:
    print(
        "[OK] No se encontraron filas duplicadas exactas. "
        "No corresponde eliminar registros."
    )
else:
    print(
        f"[REVISAR] Se encontraron {duplicados_exactos} "
        "filas duplicadas exactas."
    )

if df_crudo["UDI"].is_unique and df_crudo["Product ID"].is_unique:
    print("[OK] Los dos identificadores son únicos.")
else:
    print("[REVISAR] Existen identificadores repetidos.")

### 2.3 Categorías y valores permitidos

Las variables categóricas deben contener únicamente los valores declarados en
el diccionario de datos.

Para `Type` se esperan las categorías `L`, `M` y `H`. Las variables de falla
deben ser binarias, por lo que solo pueden contener `0` y `1`.

Se utilizará una función que compare los valores observados con los permitidos.
Si aparece un valor diferente, quedará registrado como inesperado antes de
realizar cualquier transformación.

In [ ]:
REGLAS_CATEGORICAS = {
    "Type": {"L", "M", "H"},
    "Machine failure": {0, 1},
    "TWF": {0, 1},
    "HDF": {0, 1},
    "PWF": {0, 1},
    "OSF": {0, 1},
    "RNF": {0, 1},
}


def revisar_valores_permitidos(datos, reglas):
    """
    Compara los valores observados con los valores permitidos.
    """
    resultados = []

    for columna, permitidos in reglas.items():
        if columna not in datos.columns:
            resultados.append({
                "variable": columna,
                "observados": "",
                "inesperados": "columna ausente",
                "estado": "ERROR",
            })
            continue

        observados = set(
            datos[columna].dropna().unique().tolist()
        )
        inesperados = observados - permitidos

        resultados.append({
            "variable": columna,
            "observados": ", ".join(
                sorted(map(str, observados))
            ),
            "inesperados": ", ".join(
                sorted(map(str, inesperados))
            ),
            "estado": "OK" if not inesperados else "REVISAR",
        })

    return pd.DataFrame(resultados)


VALIDACION_CATEGORIAS = revisar_valores_permitidos(
    df_crudo,
    REGLAS_CATEGORICAS,
)

VALIDACION_CATEGORIAS

Además de comprobar los valores permitidos, se contará cuántas observaciones
pertenecen a cada tipo de producto. Esto permite verificar si las tres
categorías tienen representación en el conjunto.

In [ ]:
CONTEO_TYPE = (
    df_crudo["Type"]
    .value_counts(dropna=False)
    .rename_axis("Type")
    .reset_index(name="cantidad")
)

CONTEO_TYPE["porcentaje"] = (
    CONTEO_TYPE["cantidad"] / len(df_crudo) * 100
).round(2)

CONTEO_TYPE

### 2.4 Distribución de la variable objetivo

`Machine failure` es la variable objetivo del proyecto:

- `0`: no se registró una falla.
- `1`: se registró una falla.

Antes de preparar los datos es necesario conocer la proporción de ambas clases.
Un desbalance fuerte puede hacer que métricas como la exactitud entreguen una
impresión engañosa del desempeño de un modelo.

En esta fase solo se medirá y documentará el desbalance. Las estrategias para
tratarlo deberán evaluarse posteriormente sin modificar arbitrariamente la
distribución original.

In [ ]:
def resumir_objetivo(datos, columna):
    """
    Calcula frecuencias y porcentajes de una variable objetivo.
    """
    if columna not in datos.columns:
        raise KeyError(
            f"La columna objetivo '{columna}' no existe."
        )

    conteo = (
        datos[columna]
        .value_counts(dropna=False)
        .sort_index()
        .rename_axis("valor")
        .reset_index(name="cantidad")
    )

    conteo["porcentaje"] = (
        conteo["cantidad"] / len(datos) * 100
    ).round(2)

    return conteo


RESUMEN_OBJETIVO = resumir_objetivo(
    df_crudo,
    "Machine failure",
)

TASA_FALLA = float(
    df_crudo["Machine failure"].mean() * 100
)

CLASE_MAYORITARIA = float(
    RESUMEN_OBJETIVO["porcentaje"].max()
)

print("Porcentaje de fallas:", round(TASA_FALLA, 2), "%")
print(
    "Exactitud de referencia si siempre se predijera "
    "la clase mayoritaria:",
    round(CLASE_MAYORITARIA, 2),
    "%",
)

RESUMEN_OBJETIVO

La siguiente visualización permite comparar directamente la cantidad de
registros con y sin falla. El gráfico se utiliza como evidencia de exploración;
todavía no representa el resultado de un modelo predictivo.

In [ ]:
etiquetas = ["Sin falla (0)", "Con falla (1)"]
cantidades = (
    RESUMEN_OBJETIVO
    .set_index("valor")
    .reindex([0, 1])["cantidad"]
)

fig, ax = plt.subplots(figsize=(7, 4))

barras = ax.bar(
    etiquetas,
    cantidades,
    color=["#4C78A8", "#E45756"],
)

ax.bar_label(
    barras,
    labels=[
        f"{cantidad:,}"
        for cantidad in cantidades
    ],
    padding=3,
)

ax.set_title("Distribución de Machine failure")
ax.set_xlabel("Clase")
ax.set_ylabel("Cantidad de registros")
ax.set_ylim(0, cantidades.max() * 1.12)

plt.tight_layout()
plt.show()

### Interpretación

El conjunto contiene 9.661 registros sin falla y 339 registros con falla. Por
lo tanto, las fallas representan el 3,39 % de las observaciones.

La variable objetivo presenta un desbalance considerable. Un procedimiento que
clasificara todos los registros como “sin falla” alcanzaría una exactitud de
96,61 %, aunque no identificaría ninguna falla real. Por esta razón, en etapas
posteriores no será suficiente evaluar el desempeño únicamente mediante
exactitud; también deberán considerarse métricas como sensibilidad, precisión,
F1 y el área bajo la curva ROC.

En F2 se conservará la distribución original. Cualquier estrategia de balanceo
deberá aplicarse posteriormente solo sobre los datos de entrenamiento para
evitar alterar la evaluación.

### 2.5 Rangos y valores atípicos de las variables operacionales

Las variables operacionales se encuentran en unidades y escalas diferentes.
Antes de transformarlas se revisarán sus valores mínimos, máximos, medias,
medianas y dispersión.

También se utilizará el criterio del rango intercuartílico para identificar
valores alejados del centro de cada distribución. Un valor atípico no se
considerará automáticamente un error, porque una condición operacional extrema
podría estar relacionada precisamente con una falla.

In [ ]:
COLUMNAS_OPERACIONALES = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
]

ESTADISTICAS_OPERACIONALES = (
    df_crudo[COLUMNAS_OPERACIONALES]
    .describe()
    .T
    .rename(columns={
        "count": "cantidad",
        "mean": "media",
        "std": "desviacion",
        "min": "minimo",
        "25%": "q1",
        "50%": "mediana",
        "75%": "q3",
        "max": "maximo",
    })
)

ESTADISTICAS_OPERACIONALES[
    [
        "cantidad",
        "media",
        "desviacion",
        "minimo",
        "q1",
        "mediana",
        "q3",
        "maximo",
    ]
].round(2)

El rango intercuartílico se calcula como `Q3 - Q1`. Con el criterio IQR se
considera potencialmente atípico un valor inferior a `Q1 - 1,5 × IQR` o
superior a `Q3 + 1,5 × IQR`.

Este criterio sirve para medir y describir observaciones extremas. No demuestra
que un registro sea incorrecto ni justifica por sí solo su eliminación.

In [ ]:
def detectar_atipicos_iqr(serie, factor=1.5):
    """
    Resume los valores potencialmente atípicos según el criterio IQR.
    """
    valores = serie.dropna()

    if valores.empty:
        raise ValueError(
            f"La variable '{serie.name}' no contiene valores válidos."
        )

    q1 = float(valores.quantile(0.25))
    q3 = float(valores.quantile(0.75))
    iqr = q3 - q1

    limite_inferior = q1 - factor * iqr
    limite_superior = q3 + factor * iqr

    mascara = (
        (serie < limite_inferior)
        | (serie > limite_superior)
    ).fillna(False)

    return {
        "variable": serie.name,
        "limite_inferior": limite_inferior,
        "limite_superior": limite_superior,
        "cantidad_atipicos": int(mascara.sum()),
        "porcentaje_atipicos": float(
            mascara.mean() * 100
        ),
    }


RESULTADOS_ATIPICOS = pd.DataFrame([
    detectar_atipicos_iqr(df_crudo[columna])
    for columna in COLUMNAS_OPERACIONALES
])

RESULTADOS_ATIPICOS.round(2)

### El diccionario de variables

El diccionario elaborado en F1 se reutiliza como base para mantener coherencia
entre ambas fases. En F2 se completa con el tipo observado, la cantidad de
valores únicos, el porcentaje de nulos y el tratamiento que recibirá cada
variable.

El tipo almacenado no determina por sí solo el rol analítico. Por ejemplo,
`UDI` se almacena como número entero, pero funciona como identificador y no como
una medición operacional.

In [ ]:
# Cargar el diccionario elaborado en F1.
archivo_diccionario_f1 = (
    RAIZ / "F1" / "docs" / "diccionario_variables.csv"
)

if not archivo_diccionario_f1.is_file():
    raise FileNotFoundError(
        f"No se encontró el diccionario de F1: {archivo_diccionario_f1}"
    )

diccionario = pd.read_csv(
    archivo_diccionario_f1,
    encoding="utf-8-sig",
)

# Confirmar que el diccionario cubre todas las variables observadas.
if set(diccionario["variable"]) != set(df_crudo.columns):
    raise ValueError(
        "Las variables del diccionario no coinciden con el dataset."
    )

# Tratamiento previsto para cada variable durante F2.
tratamientos = {
    "UDI": "Excluir de los predictores",
    "Product ID": "Excluir de los predictores",
    "Type": "Codificación One-Hot",
    "Air temperature [K]": "Evaluar escalamiento",
    "Process temperature [K]": "Evaluar escalamiento",
    "Rotational speed [rpm]": "Evaluar escalamiento",
    "Torque [Nm]": "Evaluar escalamiento",
    "Tool wear [min]": "Evaluar escalamiento",
    "Machine failure": "Conservar como objetivo",
    "TWF": "Excluir por fuga de información",
    "HDF": "Excluir por fuga de información",
    "PWF": "Excluir por fuga de información",
    "OSF": "Excluir por fuga de información",
    "RNF": "Excluir por fuga de información",
}

# Completar el diccionario con evidencia observada.
diccionario["tipo_observado"] = [
    str(df_crudo[variable].dtype)
    for variable in diccionario["variable"]
]

diccionario["valores_unicos"] = [
    int(df_crudo[variable].nunique())
    for variable in diccionario["variable"]
]

diccionario["porcentaje_nulos"] = [
    round(df_crudo[variable].isna().mean() * 100, 2)
    for variable in diccionario["variable"]
]

diccionario["tratamiento_f2"] = [
    tratamientos[variable]
    for variable in diccionario["variable"]
]

# Guardar el diccionario propio de F2.
archivo_diccionario_f2 = (
    DIR_DOCS / "diccionario_variables.csv"
)

diccionario.to_csv(
    archivo_diccionario_f2,
    index=False,
    encoding="utf-8-sig",
)

print("Diccionario guardado en:", archivo_diccionario_f2)
print("Variables documentadas:", len(diccionario))

diccionario

## 3. Limpieza de datos

La exploración mostró que el conjunto no contiene valores faltantes ni filas
duplicadas exactas. Por este motivo no se aplicará imputación ni se eliminarán
registros.

La limpieza se concentrará en:

- Normalizar el texto de `Type` y convertirlo en variable categórica.
- Convertir `Machine failure` a un tipo entero binario de menor tamaño.
- Excluir `UDI` y `Product ID`, porque son identificadores.
- Excluir `TWF`, `HDF`, `PWF`, `OSF` y `RNF` del conjunto analítico para evitar
  fuga de información.

Las columnas excluidas continuarán disponibles en `df_crudo` para fines de
trazabilidad. La limpieza se aplicará sobre una copia.

In [ ]:
COLUMNAS_EXCLUIDAS = [
    "UDI",
    "Product ID",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF",
]

COLUMNAS_ANALISIS = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Machine failure",
]


def limpiar_datos(datos):
    """
    Limpia y selecciona las variables necesarias para el análisis.
    """
    datos_limpios = datos.copy(deep=True)

    # El conjunto debería llegar sin valores faltantes.
    nulos = int(datos_limpios.isna().sum().sum())
    if nulos > 0:
        raise ValueError(
            f"Se encontraron {nulos} valores faltantes no esperados."
        )

    # Eliminar filas duplicadas exactas si aparecieran.
    filas_iniciales = len(datos_limpios)
    datos_limpios = datos_limpios.drop_duplicates()
    duplicados_eliminados = filas_iniciales - len(datos_limpios)

    # Normalizar y validar la variable Type.
    datos_limpios["Type"] = (
        datos_limpios["Type"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    categorias_observadas = set(
        datos_limpios["Type"].dropna().unique()
    )
    categorias_permitidas = {"L", "M", "H"}

    if not categorias_observadas.issubset(categorias_permitidas):
        inesperadas = (
            categorias_observadas - categorias_permitidas
        )
        raise ValueError(
            f"Categorías inesperadas en Type: {inesperadas}"
        )

    # Conservar solo las variables definidas para el análisis.
    datos_limpios = datos_limpios[COLUMNAS_ANALISIS].copy()

    # Casting: asignar tipos coherentes con el rol analítico.
    datos_limpios["Type"] = datos_limpios["Type"].astype("category")
    datos_limpios["Machine failure"] = (
        datos_limpios["Machine failure"].astype("int8")
    )

    print("Filas iniciales:", filas_iniciales)
    print("Duplicados eliminados:", duplicados_eliminados)
    print("Filas finales:", len(datos_limpios))
    print("Columnas finales:", datos_limpios.shape[1])

    return datos_limpios


df_limpio = limpiar_datos(df_crudo)

print("\nTipos después de la limpieza:")
print(df_limpio.dtypes)

df_limpio.head()

## 4. Transformación: codificación One-Hot

La variable `Type` contiene las categorías `L`, `M` y `H`. Como los algoritmos
trabajan con números, se transformará en tres columnas binarias mediante
One-Hot Encoding.

Este método representa la categoría sin asignarle una distancia numérica
artificial. Los nombres de las columnas serán generados directamente por
`OneHotEncoder`.

In [ ]:
def codificar_one_hot(datos, columna):
    """Codifica una variable categórica mediante One-Hot Encoding."""
    if columna not in datos.columns:
        raise KeyError(f"La columna '{columna}' no existe.")

    codificador = OneHotEncoder(
        sparse_output=False,
        handle_unknown="error",
        dtype=np.int8,
    )

    matriz = codificador.fit_transform(datos[[columna]])
    nombres = codificador.get_feature_names_out([columna])

    codificadas = pd.DataFrame(
        matriz,
        columns=nombres,
        index=datos.index,
    )

    resultado = pd.concat(
        [
            datos.drop(columns=[columna]),
            codificadas,
        ],
        axis=1,
    )

    return resultado, codificador, nombres.tolist()


df_transformado, codificador_type, COLUMNAS_TYPE = (
    codificar_one_hot(df_limpio, "Type")
)

assert (
    df_transformado[COLUMNAS_TYPE].sum(axis=1) == 1
).all()

print("Columnas creadas:", COLUMNAS_TYPE)
print("Dimensiones:", df_transformado.shape)

df_transformado.head()

## 5. Escalamiento (estandarización)

Las variables operacionales utilizan escalas diferentes. Por ejemplo, la
temperatura se encuentra cerca de 300 K, mientras que la velocidad puede superar
las 2.000 rpm. Algunos algoritmos podrían dar mayor importancia a una variable
solo porque sus números son mayores.

Se compararán tres métodos:

| Método | Transformación |
|---|---|
| StandardScaler | Media 0 y desviación estándar 1. |
| MinMaxScaler | Valores entre 0 y 1. |
| RobustScaler | Mediana 0 y rango intercuartílico 1. |

Antes de ajustar los escaladores, los datos se dividirán en entrenamiento y
prueba. El escalador aprenderá únicamente desde el conjunto de entrenamiento
para evitar fuga de información.

In [ ]:
from sklearn.model_selection import train_test_split

# Separar predictores y variable objetivo.
X = df_transformado.drop(columns=["Machine failure"])
y = df_transformado["Machine failure"]

# División reproducible y estratificada.
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=SEMILLA,
        stratify=y,
    )
)

ESCALADORES = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
}

comparacion = []

for nombre, escalador in ESCALADORES.items():
    valores = escalador.fit_transform(
        X_entrenamiento[COLUMNAS_OPERACIONALES]
    )

    tabla = pd.DataFrame(
        valores,
        columns=COLUMNAS_OPERACIONALES,
    )

    for columna in COLUMNAS_OPERACIONALES:
        comparacion.append({
            "metodo": nombre,
            "variable": columna,
            "media": tabla[columna].mean(),
            "mediana": tabla[columna].median(),
            "desviacion": tabla[columna].std(ddof=0),
            "minimo": tabla[columna].min(),
            "maximo": tabla[columna].max(),
            "iqr": (
                tabla[columna].quantile(0.75)
                - tabla[columna].quantile(0.25)
            ),
        })

COMPARACION_ESCALADORES = pd.DataFrame(comparacion)

print("Entrenamiento:", X_entrenamiento.shape)
print("Prueba:", X_prueba.shape)

COMPARACION_ESCALADORES.round(2)

La exploración mediante IQR mostró valores operacionales alejados del centro de
sus distribuciones. Por esta razón se selecciona `RobustScaler`, que utiliza la
mediana y el rango intercuartílico y resulta menos sensible a valores extremos.

El escalador se ajustará únicamente con el conjunto de entrenamiento. Después,
la misma transformación aprendida se aplicará al conjunto de prueba.

In [ ]:
escalador_final = RobustScaler()

# Crear copias para conservar las versiones anteriores.
X_entrenamiento_escalado = X_entrenamiento.copy()
X_prueba_escalado = X_prueba.copy()

# Aprender desde entrenamiento y transformarlo.
X_entrenamiento_escalado[COLUMNAS_OPERACIONALES] = (
    escalador_final.fit_transform(
        X_entrenamiento[COLUMNAS_OPERACIONALES]
    )
)

# Aplicar a prueba lo aprendido en entrenamiento.
X_prueba_escalado[COLUMNAS_OPERACIONALES] = (
    escalador_final.transform(
        X_prueba[COLUMNAS_OPERACIONALES]
    )
)

# Volver a unir los predictores con la variable objetivo.
df_entrenamiento_procesado = (
    X_entrenamiento_escalado.copy()
)
df_entrenamiento_procesado["Machine failure"] = (
    y_entrenamiento
)

df_prueba_procesado = X_prueba_escalado.copy()
df_prueba_procesado["Machine failure"] = y_prueba

# Verificar las propiedades de RobustScaler en entrenamiento.
medianas = X_entrenamiento_escalado[
    COLUMNAS_OPERACIONALES
].median()

iqr = (
    X_entrenamiento_escalado[COLUMNAS_OPERACIONALES].quantile(0.75)
    - X_entrenamiento_escalado[COLUMNAS_OPERACIONALES].quantile(0.25)
)

assert np.allclose(medianas, 0, atol=1e-8)
assert np.allclose(iqr, 1, atol=1e-8)

print("Entrenamiento procesado:", df_entrenamiento_procesado.shape)
print("Prueba procesada:", df_prueba_procesado.shape)
print("[OK] Medianas cercanas a 0.")
print("[OK] Rangos intercuartílicos cercanos a 1.")

df_entrenamiento_procesado.head()

## 6. Validación técnica y verificación del código

La validación comprobará que los conjuntos procesados no tengan valores nulos,
contengan únicamente variables numéricas, conserven la variable objetivo y
mantengan una sola categoría `Type` activa por fila.

También se probarán casos normales, límite y de excepción. Una prueba de
excepción introduce deliberadamente una entrada incorrecta y comprueba que el
programa la detecte.

In [ ]:
def validar_dataset(datos, nombre):
    """Valida la estructura e integridad de un dataset procesado."""
    assert not datos.empty, f"{nombre} está vacío."
    assert datos.isna().sum().sum() == 0, (
        f"{nombre} contiene valores nulos."
    )

    no_numericas = datos.select_dtypes(
        exclude=[np.number]
    ).columns.tolist()

    assert not no_numericas, (
        f"{nombre} contiene columnas no numéricas: {no_numericas}"
    )

    columnas_esperadas = set(
        COLUMNAS_OPERACIONALES
        + COLUMNAS_TYPE
        + ["Machine failure"]
    )

    assert set(datos.columns) == columnas_esperadas, (
        f"{nombre} no contiene las columnas esperadas."
    )

    assert set(datos["Machine failure"].unique()).issubset({0, 1}), (
        f"{nombre} contiene valores inválidos en Machine failure."
    )

    assert (
        datos[COLUMNAS_TYPE].sum(axis=1) == 1
    ).all(), f"{nombre} tiene una codificación Type incoherente."

    print(
        f"[OK] {nombre}: "
        f"{datos.shape[0]} filas × {datos.shape[1]} columnas"
    )

    return True


validar_dataset(
    df_entrenamiento_procesado,
    "entrenamiento",
)

validar_dataset(
    df_prueba_procesado,
    "prueba",
)

assert set(df_entrenamiento_procesado.index).isdisjoint(
    set(df_prueba_procesado.index)
), "Entrenamiento y prueba comparten filas."

assert (
    len(df_entrenamiento_procesado)
    + len(df_prueba_procesado)
    == len(df_transformado)
), "La división perdió o agregó registros."

print("[OK] Entrenamiento y prueba no comparten filas.")
print("[OK] Se conservaron los 10.000 registros.")


In [ ]:
# Caso normal: ambos datasets cumplen las validaciones.
assert validar_dataset(
    df_entrenamiento_procesado,
    "caso normal",
)

# Caso límite: transformar un único registro de tipo L.
caso_limite = pd.DataFrame({"Type": ["L"]})
resultado_limite = codificador_type.transform(caso_limite)

assert resultado_limite.shape == (1, 3)
assert resultado_limite.sum() == 1
print("[OK] Caso límite: un registro fue codificado correctamente.")

# Excepción 1: solicitar una columna inexistente.
try:
    codificar_one_hot(
        df_limpio,
        "columna_inexistente",
    )
    raise AssertionError(
        "La columna inexistente no produjo una excepción."
    )
except KeyError as error:
    print("[OK] Excepción capturada:", error)

# Excepción 2: introducir un valor faltante no esperado.
datos_con_nulo = df_crudo.head(3).copy()
datos_con_nulo.loc[
    datos_con_nulo.index[0],
    "Torque [Nm]",
] = np.nan

try:
    limpiar_datos(datos_con_nulo)
    raise AssertionError(
        "El valor faltante no produjo una excepción."
    )
except ValueError as error:
    print("[OK] Excepción capturada:", error)

### Visualización de control

El boxplot permite comprobar visualmente que las variables operacionales quedaron
centradas en una escala comparable. Los puntos extremos se conservan porque
pueden representar condiciones reales de operación asociadas con fallas.

In [ ]:
ax = X_entrenamiento_escalado[
    COLUMNAS_OPERACIONALES
].boxplot(
    figsize=(12, 5),
    rot=20,
)

ax.set_title(
    "Variables operacionales después de RobustScaler"
)
ax.set_ylabel("Valor escalado")

plt.tight_layout()
plt.show()

## 7. Recursividad: aplanar los metadatos del proyecto

Los metadatos se organizan como un diccionario con varios niveles. Para
exportarlos posteriormente como una tabla, se utilizará una función recursiva.

La función vuelve a llamarse cuando encuentra otro diccionario y termina cuando
encuentra un valor simple, como texto, números o valores booleanos.

In [ ]:
METADATOS = {
    "proyecto": {
        "nombre": "Mantenimiento predictivo con AI4I 2020",
        "fase": "F2",
        "grupo": 9,
        "semilla": SEMILLA,
    },
    "datos": {
        "archivo": ARCHIVO_CRUDO.name,
        "hash_sha256": HASH_ARCHIVO_CRUDO,
        "filas": int(df_crudo.shape[0]),
        "columnas": int(df_crudo.shape[1]),
        "fuente": {
            "repositorio": "UCI Machine Learning Repository",
            "doi": "10.24432/C5HS5C",
        },
    },
    "decisiones": {
        "objetivo": "Machine failure",
        "identificadores_excluidos": ["UDI", "Product ID"],
        "modos_excluidos": ["TWF", "HDF", "PWF", "OSF", "RNF"],
        "codificacion": "OneHotEncoder",
        "escalamiento": "RobustScaler",
        "division": {
            "entrenamiento": "80 %",
            "prueba": "20 %",
            "estratificada": True,
        },
    },
}


def aplanar(estructura, prefijo="", separador="."):
    """Convierte un diccionario anidado en uno de un solo nivel."""
    resultado = {}

    for clave, valor in estructura.items():
        ruta = (
            f"{prefijo}{separador}{clave}"
            if prefijo
            else str(clave)
        )

        if isinstance(valor, dict):
            resultado.update(
                aplanar(valor, ruta, separador)
            )
        elif isinstance(valor, (list, tuple)):
            resultado[ruta] = ", ".join(map(str, valor))
        else:
            resultado[ruta] = valor

    return resultado


metadatos_planos = aplanar(METADATOS)

for clave, valor in metadatos_planos.items():
    print(f"{clave:<42} {valor}")

# Casos normal, profundo, vacío y con lista.
assert aplanar({"a": 1}) == {"a": 1}
assert aplanar({"a": {"b": 2}}) == {"a.b": 2}
assert aplanar({"a": {"b": {"c": 3}}}) == {"a.b.c": 3}
assert aplanar({}) == {}
assert aplanar({"x": [1, 2]}) == {"x": "1, 2"}

print("\n[OK] La función recursiva superó cinco pruebas.")

## 8. Persistencia y trazabilidad

Los conjuntos de entrenamiento y prueba se guardarán por separado para conservar
la división realizada antes del escalamiento. También se exportarán los
metadatos y un resumen de las transformaciones.

Después de escribir cada archivo se volverá a leer para comprobar sus
dimensiones, columnas y valores nulos. El archivo original de F1 no será
sobrescrito.

In [ ]:
ARCHIVO_ENTRENAMIENTO = (
    DIR_PROCESADO / "ai4i2020_entrenamiento_procesado.csv"
)

ARCHIVO_PRUEBA = (
    DIR_PROCESADO / "ai4i2020_prueba_procesado.csv"
)

# Guardar los conjuntos procesados.
df_entrenamiento_procesado.to_csv(
    ARCHIVO_ENTRENAMIENTO,
    index=False,
    encoding="utf-8",
)

df_prueba_procesado.to_csv(
    ARCHIVO_PRUEBA,
    index=False,
    encoding="utf-8",
)

# Releer para comprobar la escritura.
entrenamiento_releido = pd.read_csv(ARCHIVO_ENTRENAMIENTO)
prueba_releida = pd.read_csv(ARCHIVO_PRUEBA)

assert entrenamiento_releido.shape == (
    df_entrenamiento_procesado.shape
)
assert prueba_releida.shape == df_prueba_procesado.shape
assert entrenamiento_releido.isna().sum().sum() == 0
assert prueba_releida.isna().sum().sum() == 0
assert list(entrenamiento_releido.columns) == list(
    df_entrenamiento_procesado.columns
)
assert list(prueba_releida.columns) == list(
    df_prueba_procesado.columns
)

# Exportar los metadatos planos.
archivo_metadatos_csv = (
    DIR_DOCS / "metadatos_proyecto.csv"
)

pd.DataFrame(
    metadatos_planos.items(),
    columns=["clave", "valor"],
).to_csv(
    archivo_metadatos_csv,
    index=False,
    encoding="utf-8-sig",
)

# Conservar también la estructura anidada en JSON.
archivo_metadatos_json = (
    DIR_DOCS / "metadatos_proyecto.json"
)

with archivo_metadatos_json.open(
    "w",
    encoding="utf-8",
) as archivo:
    json.dump(
        METADATOS,
        archivo,
        ensure_ascii=False,
        indent=2,
    )

# Resumen de las etapas del pipeline.
resumen_transformaciones = pd.DataFrame([
    {
        "etapa": "Datos originales",
        "filas": df_crudo.shape[0],
        "columnas": df_crudo.shape[1],
        "nulos": int(df_crudo.isna().sum().sum()),
        "detalle": "Archivo original sin modificar",
    },
    {
        "etapa": "Limpieza y selección",
        "filas": df_limpio.shape[0],
        "columnas": df_limpio.shape[1],
        "nulos": int(df_limpio.isna().sum().sum()),
        "detalle": "Identificadores y modos de falla excluidos",
    },
    {
        "etapa": "Codificación One-Hot",
        "filas": df_transformado.shape[0],
        "columnas": df_transformado.shape[1],
        "nulos": int(df_transformado.isna().sum().sum()),
        "detalle": "Type transformado en tres columnas",
    },
    {
        "etapa": "Entrenamiento escalado",
        "filas": df_entrenamiento_procesado.shape[0],
        "columnas": df_entrenamiento_procesado.shape[1],
        "nulos": int(
            df_entrenamiento_procesado.isna().sum().sum()
        ),
        "detalle": "RobustScaler ajustado con entrenamiento",
    },
    {
        "etapa": "Prueba escalada",
        "filas": df_prueba_procesado.shape[0],
        "columnas": df_prueba_procesado.shape[1],
        "nulos": int(df_prueba_procesado.isna().sum().sum()),
        "detalle": "Transformada sin reajustar el escalador",
    },
])

archivo_resumen = (
    DIR_DOCS / "resumen_transformaciones.csv"
)

resumen_transformaciones.to_csv(
    archivo_resumen,
    index=False,
    encoding="utf-8-sig",
)

print("[OK] Archivos guardados y verificados:")
print("-", ARCHIVO_ENTRENAMIENTO)
print("-", ARCHIVO_PRUEBA)
print("-", archivo_metadatos_csv)
print("-", archivo_metadatos_json)
print("-", archivo_resumen)

resumen_transformaciones

## 9. Reflexión técnica

### Hallazgos

El archivo original contiene 10.000 registros y 14 variables. No se encontraron
valores faltantes, filas duplicadas exactas ni identificadores repetidos. Por
esta razón no fue necesario imputar datos ni eliminar observaciones.

La variable `Type` contiene 6.000 productos L, 2.997 productos M y 1.003
productos H. La variable objetivo presenta un desbalance considerable: solo 339
registros, equivalentes al 3,39 %, corresponden a fallas. Este resultado deberá
considerarse en la evaluación de modelos posteriores, porque la exactitud por sí
sola podría entregar una impresión engañosa.

El criterio IQR identificó 418 valores potencialmente atípicos en velocidad de
rotación y 69 en torque. Estos registros se conservaron porque podrían
representar condiciones operacionales reales asociadas con fallas. La presencia
de esos valores apoyó la elección de `RobustScaler`.

### Decisiones técnicas

Los identificadores `UDI` y `Product ID` se excluyeron porque no representan
condiciones de operación. Los indicadores `TWF`, `HDF`, `PWF`, `OSF` y `RNF`
también se excluyeron de los predictores para evitar fuga de información.

`Type` se transformó mediante One-Hot Encoding. Las variables operacionales se
escalaron con `RobustScaler`, ajustado únicamente con el conjunto de
entrenamiento. La división estratificada conservó aproximadamente la proporción
de fallas en los conjuntos de entrenamiento y prueba.

El pipeline mantuvo los 10.000 registros: 8.000 se asignaron a entrenamiento y
2.000 a prueba. Ambos conjuntos quedaron con nueve columnas numéricas y sin
valores faltantes.

### Dificultades y soluciones

La principal dificultad fue distinguir entre variables numéricas almacenadas y
variables útiles como predictores. El diccionario de variables permitió separar
identificadores, condiciones operacionales, variable objetivo y modos de falla.

También fue necesario evitar que el escalador aprendiera información del
conjunto de prueba. Esto se resolvió dividiendo los datos antes de ajustar
`RobustScaler` y aplicando posteriormente la transformación aprendida al conjunto
de prueba.

### Decisiones pendientes

En las fases posteriores deberá evaluarse si el desbalance requiere ponderación
de clases u otra estrategia aplicada únicamente sobre entrenamiento. También
será necesario seleccionar métricas adecuadas, como sensibilidad, precisión, F1
y área bajo la curva ROC.

Los resultados deben interpretarse considerando que AI4I 2020 es un conjunto
sintético. El funcionamiento del pipeline no demuestra por sí solo que los
mismos patrones se reproduzcan en maquinaria industrial real.

## 10. Verificación antes de entregar

### Notebook

- [x] Ejecuta completo después de reiniciar el kernel, sin errores.
- [x] La numeración de ejecución es continua.
- [x] La semilla está declarada.
- [x] Las rutas son reproducibles.
- [x] Los bloques de código están explicados.

### Contenido técnico

- [x] La procedencia del dataset está documentada.
- [x] El diccionario de variables está guardado.
- [x] Las decisiones de limpieza están justificadas.
- [x] La codificación y el escalamiento fueron verificados.
- [x] Hay pruebas normales, límite y de excepción.

### Archivos

- [x] El archivo original se conserva sin modificar.
- [x] Los datos procesados están en `F2/data/processed/`.
- [x] Los documentos están en `F2/docs/`.
- [x] El notebook está exportado a HTML.
- [x] El README está actualizado.

### Repositorio

- [x] Los cambios de F2 tienen un commit.
- [x] Los cambios están enviados a GitHub.